In [ ]:
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
import numpy as np
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
onemap_api_key = os.environ.get('ONEMAP_API_KEY')

if not onemap_api_key:
    raise ValueError(
        "ONEMAP_API_KEY not found. Please create a .env file in this directory with: \n"
        "ONEMAP_API_KEY=your_api_key_here"
    )

In [ ]:
# Get file names 
hdb_points_file = "../datasets/resale_2019_to_2025_cleaned.geojson"
output_file = "hdb_cbd_distances.csv"

### 1. Define CBD coordinates with Raffles Places as reference point

In [ ]:
# Define CBD coordinates with Raffles Place as the reference point
# Transformed to EPSG:3414 for distance calculation
cbd_coords = (1.2844988, 103.8513350) 
cbd_gdf = gpd.GeoDataFrame(
    {'name': ['CBD']},
    geometry=gpd.points_from_xy([cbd_coords[1]], [cbd_coords[0]]),
    crs='EPSG:4326'
)
cbd_gdf = cbd_gdf.to_crs(epsg=3414)

cbd_gdf.head()

### 2. Get euclidian distance between CBD and HDB 

In [ ]:
# read files
unique_hdb = gpd.read_file(hdb_points_file).copy()

# get unique hdb locations
unique_hdb = unique_hdb.drop_duplicates(subset=["postal_code"]).copy()
unique_hdb = unique_hdb.to_crs(epsg=3414)
cbd_gdf = cbd_gdf.to_crs(epsg=3414)

# get CBD point
cbd_point = cbd_gdf.geometry.iloc[0]

# get Euclidean distance to CBD
unique_hdb["dist_to_cbd_m"] = unique_hdb.geometry.distance(cbd_point).round(1)

# output table
cbd_distance = unique_hdb[[
    "postal_code",
    "dist_to_cbd_m"
]].copy()

print(cbd_distance.head())
print(cbd_distance.info())

In [ ]:
cbd_distance.to_csv("../cleaned_datasets/hdb_cbd_distances.csv", index=False)